# UAS Risk Analysis Visualization

## Setup

In [1]:
!python -m pip install -q geopandas seaborn s3fs pyarrow fiona

### Imports

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from glob import glob
import subprocess

### Configurations

In [2]:
sns.set_style('darkgrid')

## Load UAS Risk Analysis Results

In [11]:
risk_scores_base_path = 's3://endurasoft-dev-risk-framework/analysis/uas_risk_scores/v2/'

In [3]:
risk_scores_df = pd.read_csv(os.path.join(risk_scores_base_path, 'consolidated/uas_risk_scores_csv/uas_risk_scores.csv'), low_memory=False)
risk_scores_df.head()

,objectid,ceiling,unit,map_eff,last_edit,latitude,longitude,globalid,arpt_count,apt1_faaid,...,apt4_enabled,apt5_enabled,shape__length,shape__area,shape__area_2,shape__length_2,wkt,flight_intersections,uas_sightings_intersections,risk_score
0,12,200,Feet,11/28/2024,4/27/2017,28.712505,-96.30418,8fc5e60e-0381-4480-bbd1-4a877e6821ba,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.3083437419521 28.7083377237965...,0,0,0.0
1,14,100,Feet,11/28/2024,4/27/2017,28.712505,-96.28751,9f03cc7a-c9d1-4855-bf0e-01304ae47bc5,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.2916770723614 28.7083377237965...,0,0,0.0
2,16,0,Feet,11/28/2024,4/27/2017,28.712505,-96.27084,5cc33746-e3e2-41a4-a138-4e1ed3c57bda,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.2750104017707 28.7083377237965...,0,0,0.0
3,19,0,Feet,11/28/2024,4/27/2017,28.712505,-96.23751,2413c780-984c-4cab-ab2a-705b492dfe41,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.2416770615893 28.7083377237965...,0,0,0.0
4,23,100,Feet,11/28/2024,4/27/2017,28.712505,-96.20418,480eb259-4697-4b00-9f6e-24882645f7c7,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.208343722408 28.7083377237965 ...,0,0,0.0


In [6]:
risk_scores_df.shape

(376569, 49)

In [7]:
# Rename long target columns (columns with more than 10 characters are truncated)
risk_scores_df.rename({'flight_intersections': 'flight_int', 'uas_sightings_intersections': 'uas_int'}, axis=1, inplace=True)

## Convert to GeoPandas and Save Shapefile

In [8]:
# Convert risk scores dataframe to geopandas
risk_scores_gdf = gpd.GeoDataFrame(
    risk_scores_df,
    geometry=gpd.GeoSeries.from_wkt(risk_scores_df['wkt'])
)

In [9]:
shapefile_dir = 'uas_risk_scores_shapefile'
os.makedirs(shapefile_dir, exist_ok=True)
risk_scores_gdf.to_file(os.path.join(shapefile_dir, 'uas_risk_scores.shp'), driver='ESRI Shapefile', engine='fiona', crs='EPSG:4326')

/tmp/ipykernel_13924/343700671.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  risk_scores_gdf.to_file(os.path.join(shapefile_dir, 'uas_risk_scores.shp'), driver='ESRI Shapefile', engine='fiona', crs='EPSG:4326')


In [10]:
glob(os.path.join(shapefile_dir, '*.*'))

['uas_risk_scores_shapefile/uas_risk_scores.shx',
 'uas_risk_scores_shapefile/uas_risk_scores.shp',
 'uas_risk_scores_shapefile/uas_risk_scores.prj',
 'uas_risk_scores_shapefile/uas_risk_scores.dbf',
 'uas_risk_scores_shapefile/uas_risk_scores.cpg']

In [12]:
# Create a zip file
with zipfile.ZipFile('uas_risk_scores_shapefile.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all shapefile components to zip
    for path in glob(os.path.join(shapefile_dir, '*.*')):
        zipf.write(path)

In [13]:
# Upload to s3
shapefile_outpath = os.path.join(risk_scores_base_path, 'viz/uas_risk_scores_shapefile.zip')
subprocess.check_call(f'aws s3 cp uas_risk_scores_shapefile.zip {shapefile_outpath}'.split())

upload: ./uas_risk_scores_shapefile.zip to s3://endurasoft-dev-risk-framework/analysis/uas_risk_scores/v2/viz/uas_risk_scores_shapefile.zip


0